In [0]:
dbutils.widgets.text("file_path", "default", "enter a value")
incoming_file_path = dbutils.widgets.get("file_path")
print(incoming_file_path)

default


In [0]:
from pyspark.sql.functions import current_timestamp
from pyspark.sql.functions import col
import pyspark.sql.functions as F
import re 

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import *


df = (
    spark.read.format("csv")
    .option("header", True)
    .option("sep", ",")
    .load(incoming_file_path)
    .withColumn("ingested_timestamp", current_timestamp())
)

for c in df.columns:
    new_col = c.strip()
    df = df.withColumnRenamed(c,new_col)
    



---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7208271537908073>, line 15
      7 from pyspark.sql import functions as F
      8 from pyspark.sql.types import *
     11 df = (
     12     spark.read.format("csv")
     13     .option("header", True)
     14     .option("sep", ",")
---> 15     .load(file_path)
     16     .withColumn("ingested_timestamp", current_timestamp())
     17 )
     19 for c in df.columns:
     20     new_col = c.strip()

NameError: name 'file_path' is not defined

In [0]:
df.filter(df.FlightDate.isNull()).count()

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7208271537908076>, line 1
----> 1 df.filter(df.FlightDate.isNull()).count()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:320, in DataFrame.count(self)
    317 def count(self) -> int:
    318     table, _ = self.agg(
    319         F._invoke_function("count", F.lit(1))
--> 320     )._to_table()  # type: ignore[operator]
    321     return table[0][0].as_py()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1984, in DataFrame._to_table(self)
   1982 def _to_table(self) -> Tuple["pa.Table", Optional[StructType]]:
   1983     query = self._plan.to_proto(self._session.client)
-> 1984     table, schema, self._execution_info = self._session.client.to_table(
   1985         query, self._plan.observations
   1986     )
   1987     assert table is not N

In [0]:
critical_col = ["FlightDate","Reporting_Airline" ,"Origin","Dest" , "DepDelay" , "ArrDelay" , "Cancelled" ,"Distance"]

null_list = [F.round( (F.avg(F.col(colm).isNull().cast("int"))*100),2).alias(colm) for colm in critical_col]

df.select(null_list).show()

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5671268314822158>, line 5
      1 critical_col = ["FlightDate","Reporting_Airline" ,"Origin","Dest" , "DepDelay" , "ArrDelay" , "Cancelled" ,"Distance"]
      3 null_list = [F.round( (F.avg(F.col(colm).isNull().cast("int"))*100),2).alias(colm) for colm in critical_col]
----> 5 df.select(null_list).show()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1200, in DataFrame.show(self, n, truncate, vertical)
   1199 def show(self, n: int = 20, truncate: Union[bool, int] = True, vertical: bool = False) -> None:
-> 1200     print(self._show_string(n, truncate, vertical))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:911, in DataFrame._show_string(self, n, truncate, vertical)
    894     except ValueError:
    895         raise PySparkTypeError(
    89

In [0]:
df_bronze = df.withColumns({
    # --- Presence checks (NULL = 1, present = 0)
    "null_flight_date_dq": F.col("FlightDate").isNull().cast("int"),
    "null_airline_dq":     F.col("Reporting_Airline").isNull().cast("int"),
    "null_origin_dq":      F.col("Origin").isNull().cast("int"),
    "null_dest_dq":        F.col("Dest").isNull().cast("int")
})

In [0]:
df_bronze = df_bronze.withColumns({
    # DepDelay: values outside [-120, 1440] are suspicious
    "delay_range_dq":    F.when(F.col("Cancelled").cast("double") == 1.0 , 0 ).otherwise((~F.col("DepDelay").cast("double").between(-120, 1440)).cast("int")).cast("int"),

    # Distance: must be positive (cast to double first to handle decimal strings)
    "dq_distance_range": (~F.col("Distance").cast("double").between(1, 10000)).cast("int"),

    # Consistency checks CancellationCode should exist IFF Cancelled = 1
    "cancellation_code_dq": (
        (F.col("Cancelled").cast("double").cast("int") == 1) & F.col("CancellationCode").isNull() |
        (F.col("Cancelled").cast("double").cast("int") == 0) & F.col("CancellationCode").isNotNull()
    ).cast("int")
})

df_bronze.show()

+----+-------+-----+----------+---------+----------+-----------------+------------------------+---------------------------+-----------+-------------------------------+---------------+------------------+------------------+------+---------------+-----------+---------------+---------------+---------+-------------+----------------+----------------+----+-------------+---------+-------------+-------------+-------+----------+-------+--------+---------------+--------+--------------------+----------+-------+---------+--------+------+----------+-------+--------+---------------+--------+------------------+----------+---------+----------------+--------+--------------+-----------------+-------+-------+--------+-------------+------------+------------+--------+-------------+-----------------+------------+-------------+---------------+------------------+--------------+--------------------+-----------+-----------+-----------+-------------+----------------+------------+--------------+----------------+--

In [0]:
df_bronze = df_bronze.withColumn("same_origin_dest_dq",((F.col("Origin") == F.col("Dest") ).cast("Double")))

In [0]:
df_bronze = df_bronze.withColumns({
    "year": F.year("FlightDate").cast("string"),
    "month": F.lpad(F.month("FlightDate").cast("string"), 2, "0"),
})

In [0]:
df_bronze_cleaned = df_bronze


In [0]:
li = []
for i in df_bronze_cleaned.columns:
    if i.endswith("_dq") or i.endswith("_range"):

        li.append(i)

print(li)


['null_flight_date_dq', 'null_airline_dq', 'null_origin_dq', 'null_dest_dq', 'delay_range_dq', 'dq_distance_range', 'cancellation_code_dq', 'same_origin_dest_dq']


In [0]:
df_bronze_cleaned = df_bronze_cleaned.withColumn("Total_flag" , sum(F.col(i) for i in li) )

In [0]:
df_bronze_cleaned= df_bronze_cleaned.withColumn("dq_is_cleaned" , (F.col("Total_flag") == 0 ).cast("int") )

In [0]:
new_df = df_bronze_cleaned

In [0]:
new_df = new_df.dropDuplicates(
    ["FlightDate", "Reporting_Airline", "Origin", "Dest", "Distance"]
)

In [0]:
new_df.display()

year,Quarter,month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,IATA_CODE_Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,OriginAirportID,OriginAirportSeqID,OriginCityMarketID,Origin,OriginCityName,OriginState,OriginStateFips,OriginStateName,OriginWac,DestAirportID,DestAirportSeqID,DestCityMarketID,Dest,DestCityName,DestState,DestStateFips,DestStateName,DestWac,CRSDepTime,DepTime,DepDelay,DepDelayMinutes,DepDel15,DepartureDelayGroups,DepTimeBlk,TaxiOut,WheelsOff,WheelsOn,TaxiIn,CRSArrTime,ArrTime,ArrDelay,ArrDelayMinutes,ArrDel15,ArrivalDelayGroups,ArrTimeBlk,Cancelled,CancellationCode,Diverted,CRSElapsedTime,ActualElapsedTime,AirTime,Flights,Distance,DistanceGroup,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay,FirstDepTime,TotalAddGTime,LongestAddGTime,DivAirportLandings,DivReachedDest,DivActualElapsedTime,DivArrDelay,DivDistance,Div1Airport,Div1AirportID,Div1AirportSeqID,Div1WheelsOn,Div1TotalGTime,Div1LongestGTime,Div1WheelsOff,Div1TailNum,Div2Airport,Div2AirportID,Div2AirportSeqID,Div2WheelsOn,Div2TotalGTime,Div2LongestGTime,Div2WheelsOff,Div2TailNum,Div3Airport,Div3AirportID,Div3AirportSeqID,Div3WheelsOn,Div3TotalGTime,Div3LongestGTime,Div3WheelsOff,Div3TailNum,Div4Airport,Div4AirportID,Div4AirportSeqID,Div4WheelsOn,Div4TotalGTime,Div4LongestGTime,Div4WheelsOff,Div4TailNum,Div5Airport,Div5AirportID,Div5AirportSeqID,Div5WheelsOn,Div5TotalGTime,Div5LongestGTime,Div5WheelsOff,Div5TailNum,_rescued_data,ingested_timestamp,null_flight_date_dq,null_airline_dq,null_origin_dq,null_dest_dq,delay_range_dq,dq_distance_range,cancellation_code_dq,same_origin_dest_dq,Total_flag,dq_is_cleaned
2026,1,01,4,7,2026-01-04,WN,19393,WN,N1803U,3469,12954,1295407,32575,LGB,"Long Beach, CA",CA,06,California,91,11292,1129203,30325,DEN,"Denver, CO",CO,08,Colorado,82,1135,1137,2.00,2.00,0.00,0,1100-1159,17.00,1154,1435,6.00,1450,1441,-9.00,0.00,0.00,-1,1400-1459,0.00,null,0.00,135.00,124.00,101.00,1.00,854.00,4,null,null,null,null,null,null,null,null,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"{""_c109"":"""",""_file_path"":""/Volumes/us_project_catalog/raw/raw_volume/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2026_1.csv""}",2026-09-26T14:37:18.359Z,0,0,0,0,0,0,0,0.0,0.0,1
2026,1,01,4,7,2026-01-04,WN,19393,WN,N8948Q,852,12954,1295407,32575,LGB,"Long Beach, CA",CA,06,California,91,14893,1489302,33192,SMF,"Sacramento, CA",CA,06,California,91,1640,1657,17.00,17.00,1.00,1,1600-1659,11.00,1708,1813,3.00,1810,1816,6.00,6.00,0.00,0,1800-1859,0.00,null,0.00,90.00,79.00,65.00,1.00,387.00,2,null,null,null,null,null,null,null,null,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"{""_c109"":"""",""_file_path"":""/Volumes/us_project_catalog/raw/raw_volume/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2026_1.csv""}",2026-09-26T14:37:18.359Z,0,0,0,0,0,0,0,0.0,0.0,1
2026,1,01,4,7,2026-01-04,WN,19393,WN,N495WN,1253,13204,1320403,31454,MCO,"Orlando, FL",FL,12,Florida,33,10397,1039708,30397,ATL,"Atlanta, GA",GA,13,Georgia,34,1500,1515,15.00,15.00,1.00,1,1500-1559,26.00,1541,1643,5.00,1635,1648,13.00,13.00,0.00,0,1600-1659,0.00,null,0.00,95.00,93.00,62.00,1.00,404.00,2,null,null,null,null,null,null,null,null,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"{""_c109"":"""",""_file_path"":""/Volumes/us_project_catalog/raw/raw_volume/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2026_1.csv""}",2026-09-26T14:37:18.359Z,0,0,0,0,0,0,0,0.0,0.0,1
2026

In [0]:
new_df =  new_df.withColumn("is_Weekend" , ((F.col("DayOfWeek") == 7) | (F.col("DayOfWeek") == 6)).cast("int")) 

In [0]:
df_bronze_cleaned =  new_df.withColumn("Delay_cateory",
    F.when(F.col("ArrDelay").cast("double") < 0, "Early")
    .when((F.col("ArrDelay").cast("double") >= 0) & (F.col("ArrDelay").cast("double") <= 15), "On Time")
    .when((F.col("ArrDelay").cast("double") >= 16) & (F.col("ArrDelay").cast("double") <= 45), "Minor_delay")
    .when((F.col("ArrDelay").cast("double") >= 46) & (F.col("ArrDelay").cast("double") <= 120), "Major_delay")
    .otherwise("Unknown_delay")
)

In [0]:
df_bronze_cleaned = df_bronze_cleaned.withColumns( 
{
"CRSArrTime" : ((F.col("CRSArrTime").cast("double") / 100) *60) + ((F.col("CRSArrTime") % 100).cast("int") ),
"ArrTime" : ((F.col("ArrTime").cast("double") / 100) *60) + ((F.col("ArrTime") % 100).cast("int") )
}) 

In [0]:
delayli = ['CarrierDelay','WeatherDelay','NASDelay', 'SecurityDelay', 'LateAircraftDelay']

try_df = df_bronze_cleaned.withColumn("Sum_delay_cause" , sum(F.col(i).cast("double") for i in delayli) )

try_df.display()

year,Quarter,month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,IATA_CODE_Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,OriginAirportID,OriginAirportSeqID,OriginCityMarketID,Origin,OriginCityName,OriginState,OriginStateFips,OriginStateName,OriginWac,DestAirportID,DestAirportSeqID,DestCityMarketID,Dest,DestCityName,DestState,DestStateFips,DestStateName,DestWac,CRSDepTime,DepTime,DepDelay,DepDelayMinutes,DepDel15,DepartureDelayGroups,DepTimeBlk,TaxiOut,WheelsOff,WheelsOn,TaxiIn,CRSArrTime,ArrTime,ArrDelay,ArrDelayMinutes,ArrDel15,ArrivalDelayGroups,ArrTimeBlk,Cancelled,CancellationCode,Diverted,CRSElapsedTime,ActualElapsedTime,AirTime,Flights,Distance,DistanceGroup,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay,FirstDepTime,TotalAddGTime,LongestAddGTime,DivAirportLandings,DivReachedDest,DivActualElapsedTime,DivArrDelay,DivDistance,Div1Airport,Div1AirportID,Div1AirportSeqID,Div1WheelsOn,Div1TotalGTime,Div1LongestGTime,Div1WheelsOff,Div1TailNum,Div2Airport,Div2AirportID,Div2AirportSeqID,Div2WheelsOn,Div2TotalGTime,Div2LongestGTime,Div2WheelsOff,Div2TailNum,Div3Airport,Div3AirportID,Div3AirportSeqID,Div3WheelsOn,Div3TotalGTime,Div3LongestGTime,Div3WheelsOff,Div3TailNum,Div4Airport,Div4AirportID,Div4AirportSeqID,Div4WheelsOn,Div4TotalGTime,Div4LongestGTime,Div4WheelsOff,Div4TailNum,Div5Airport,Div5AirportID,Div5AirportSeqID,Div5WheelsOn,Div5TotalGTime,Div5LongestGTime,Div5WheelsOff,Div5TailNum,_rescued_data,ingested_timestamp,null_flight_date_dq,null_airline_dq,null_origin_dq,null_dest_dq,delay_range_dq,dq_distance_range,cancellation_code_dq,same_origin_dest_dq,Total_flag,dq_is_cleaned,is_Weekend,Delay_cateory,Sum_delay_cause
2026,1,01,4,7,2026-01-04,WN,19393,WN,N1803U,3469,12954,1295407,32575,LGB,"Long Beach, CA",CA,06,California,91,11292,1129203,30325,DEN,"Denver, CO",CO,08,Colorado,82,1135,1137,2.00,2.00,0.00,0,1100-1159,17.00,1154,1435,6.00,920.0,905.6,-9.00,0.00,0.00,-1,1400-1459,0.00,null,0.00,135.00,124.00,101.00,1.00,854.00,4,null,null,null,null,null,null,null,null,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"{""_c109"":"""",""_file_path"":""/Volumes/us_project_catalog/raw/raw_volume/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2026_1.csv""}",2026-09-26T14:37:18.359Z,0,0,0,0,0,0,0,0.0,0.0,1,1,Early,null
2026,1,01,4,7,2026-01-04,WN,19393,WN,N8948Q,852,12954,1295407,32575,LGB,"Long Beach, CA",CA,06,California,91,14893,1489302,33192,SMF,"Sacramento, CA",CA,06,California,91,1640,1657,17.00,17.00,1.00,1,1600-1659,11.00,1708,1813,3.00,1096.0,1105.6,6.00,6.00,0.00,0,1800-1859,0.00,null,0.00,90.00,79.00,65.00,1.00,387.00,2,null,null,null,null,null,null,null,null,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"{""_c109"":"""",""_file_path"":""/Volumes/us_project_catalog/raw/raw_volume/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2026_1.csv""}",2026-09-26T14:37:18.359Z,0,0,0,0,0,0,0,0.0,0.0,1,1,On Time,null
2026,1,01,4,7,2026-01-04,WN,19393,WN,N495WN,1253,13204,1320403,31454,MCO,"Orlando, FL",FL,12,Florida,33,10397,1039708,30397,ATL,"Atlanta, GA",GA,13,Georgia,34,1500,1515,15.00,15.00,1.00,1,1500-1559,26.00,1541,1643,5.00,1016.0000000000001,1036.8000000000002,13.00,13.00,0.00,0,1600-1659,0.00,null,0.00,95.00,93.00,62.00,1.00,404.00,2,null,null,null,null,null,null,null,null,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"{""_c109"":"""",""_file_path"":""/Volumes/us_project_catalog/raw/raw_volume/On_Time_Reporting_Carrier_

In [0]:
df_bronze_cleaned =  try_df.withColumn("Data_quality_delay_mismatch" , F.when( (F.col("cancelled") == 0.00) & (F.col("ArrDelay").isNotNull()) , F.abs( (F.col("Sum_delay_cause")) - (F.col("ArrDelay"))  ))  )

In [0]:
df_bronze_cleaned.display()

year,Quarter,month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,IATA_CODE_Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,OriginAirportID,OriginAirportSeqID,OriginCityMarketID,Origin,OriginCityName,OriginState,OriginStateFips,OriginStateName,OriginWac,DestAirportID,DestAirportSeqID,DestCityMarketID,Dest,DestCityName,DestState,DestStateFips,DestStateName,DestWac,CRSDepTime,DepTime,DepDelay,DepDelayMinutes,DepDel15,DepartureDelayGroups,DepTimeBlk,TaxiOut,WheelsOff,WheelsOn,TaxiIn,CRSArrTime,ArrTime,ArrDelay,ArrDelayMinutes,ArrDel15,ArrivalDelayGroups,ArrTimeBlk,Cancelled,CancellationCode,Diverted,CRSElapsedTime,ActualElapsedTime,AirTime,Flights,Distance,DistanceGroup,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay,FirstDepTime,TotalAddGTime,LongestAddGTime,DivAirportLandings,DivReachedDest,DivActualElapsedTime,DivArrDelay,DivDistance,Div1Airport,Div1AirportID,Div1AirportSeqID,Div1WheelsOn,Div1TotalGTime,Div1LongestGTime,Div1WheelsOff,Div1TailNum,Div2Airport,Div2AirportID,Div2AirportSeqID,Div2WheelsOn,Div2TotalGTime,Div2LongestGTime,Div2WheelsOff,Div2TailNum,Div3Airport,Div3AirportID,Div3AirportSeqID,Div3WheelsOn,Div3TotalGTime,Div3LongestGTime,Div3WheelsOff,Div3TailNum,Div4Airport,Div4AirportID,Div4AirportSeqID,Div4WheelsOn,Div4TotalGTime,Div4LongestGTime,Div4WheelsOff,Div4TailNum,Div5Airport,Div5AirportID,Div5AirportSeqID,Div5WheelsOn,Div5TotalGTime,Div5LongestGTime,Div5WheelsOff,Div5TailNum,_rescued_data,ingested_timestamp,null_flight_date_dq,null_airline_dq,null_origin_dq,null_dest_dq,delay_range_dq,dq_distance_range,cancellation_code_dq,same_origin_dest_dq,Total_flag,dq_is_cleaned,is_Weekend,Delay_cateory,Sum_delay_cause,Data_quality_delay_mismatch
2026,1,01,4,7,2026-01-04,WN,19393,WN,N1803U,3469,12954,1295407,32575,LGB,"Long Beach, CA",CA,06,California,91,11292,1129203,30325,DEN,"Denver, CO",CO,08,Colorado,82,1135,1137,2.00,2.00,0.00,0,1100-1159,17.00,1154,1435,6.00,920.0,905.6,-9.00,0.00,0.00,-1,1400-1459,0.00,null,0.00,135.00,124.00,101.00,1.00,854.00,4,null,null,null,null,null,null,null,null,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"{""_c109"":"""",""_file_path"":""/Volumes/us_project_catalog/raw/raw_volume/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2026_1.csv""}",2026-09-26T14:37:18.359Z,0,0,0,0,0,0,0,0.0,0.0,1,1,Early,null,null
2026,1,01,4,7,2026-01-04,WN,19393,WN,N8948Q,852,12954,1295407,32575,LGB,"Long Beach, CA",CA,06,California,91,14893,1489302,33192,SMF,"Sacramento, CA",CA,06,California,91,1640,1657,17.00,17.00,1.00,1,1600-1659,11.00,1708,1813,3.00,1096.0,1105.6,6.00,6.00,0.00,0,1800-1859,0.00,null,0.00,90.00,79.00,65.00,1.00,387.00,2,null,null,null,null,null,null,null,null,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"{""_c109"":"""",""_file_path"":""/Volumes/us_project_catalog/raw/raw_volume/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2026_1.csv""}",2026-09-26T14:37:18.359Z,0,0,0,0,0,0,0,0.0,0.0,1,1,On Time,null,null
2026,1,01,4,7,2026-01-04,WN,19393,WN,N495WN,1253,13204,1320403,31454,MCO,"Orlando, FL",FL,12,Florida,33,10397,1039708,30397,ATL,"Atlanta, GA",GA,13,Georgia,34,1500,1515,15.00,15.00,1.00,1,1500-1559,26.00,1541,1643,5.00,1016.0000000000001,1036.8000000000002,13.00,13.00,0.00,0,1600-1659,0.00,null,0.00,95.00,93.00,62.00,1.00,404.00,2,null,null,null,null,null,null,null,null,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"{""_c109"":"""",""_file_path"":""/Volumes/us_project_catalog/raw

In [0]:
try_df = df_bronze_cleaned.withColumn("Cancellation_category" , F.when((F.col("CancellationCode") == "A") ,"Carrier" ).when((F.col("CancellationCode") == "B") ,"Weather" ).when((F.col("CancellationCode") == "C") ,"Air_system" ).when((F.col("CancellationCode") == "D") ,"Security" ).otherwise("Unknown")  )




In [0]:
try_df.display()

year,Quarter,month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,IATA_CODE_Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,OriginAirportID,OriginAirportSeqID,OriginCityMarketID,Origin,OriginCityName,OriginState,OriginStateFips,OriginStateName,OriginWac,DestAirportID,DestAirportSeqID,DestCityMarketID,Dest,DestCityName,DestState,DestStateFips,DestStateName,DestWac,CRSDepTime,DepTime,DepDelay,DepDelayMinutes,DepDel15,DepartureDelayGroups,DepTimeBlk,TaxiOut,WheelsOff,WheelsOn,TaxiIn,CRSArrTime,ArrTime,ArrDelay,ArrDelayMinutes,ArrDel15,ArrivalDelayGroups,ArrTimeBlk,Cancelled,CancellationCode,Diverted,CRSElapsedTime,ActualElapsedTime,AirTime,Flights,Distance,DistanceGroup,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay,FirstDepTime,TotalAddGTime,LongestAddGTime,DivAirportLandings,DivReachedDest,DivActualElapsedTime,DivArrDelay,DivDistance,Div1Airport,Div1AirportID,Div1AirportSeqID,Div1WheelsOn,Div1TotalGTime,Div1LongestGTime,Div1WheelsOff,Div1TailNum,Div2Airport,Div2AirportID,Div2AirportSeqID,Div2WheelsOn,Div2TotalGTime,Div2LongestGTime,Div2WheelsOff,Div2TailNum,Div3Airport,Div3AirportID,Div3AirportSeqID,Div3WheelsOn,Div3TotalGTime,Div3LongestGTime,Div3WheelsOff,Div3TailNum,Div4Airport,Div4AirportID,Div4AirportSeqID,Div4WheelsOn,Div4TotalGTime,Div4LongestGTime,Div4WheelsOff,Div4TailNum,Div5Airport,Div5AirportID,Div5AirportSeqID,Div5WheelsOn,Div5TotalGTime,Div5LongestGTime,Div5WheelsOff,Div5TailNum,_rescued_data,ingested_timestamp,null_flight_date_dq,null_airline_dq,null_origin_dq,null_dest_dq,delay_range_dq,dq_distance_range,cancellation_code_dq,same_origin_dest_dq,Total_flag,dq_is_cleaned,is_Weekend,Delay_cateory,Sum_delay_cause,Data_quality_delay_mismatch,Cancellation_category
2026,1,01,4,7,2026-01-04,WN,19393,WN,N1803U,3469,12954,1295407,32575,LGB,"Long Beach, CA",CA,06,California,91,11292,1129203,30325,DEN,"Denver, CO",CO,08,Colorado,82,1135,1137,2.00,2.00,0.00,0,1100-1159,17.00,1154,1435,6.00,920.0,905.6,-9.00,0.00,0.00,-1,1400-1459,0.00,null,0.00,135.00,124.00,101.00,1.00,854.00,4,null,null,null,null,null,null,null,null,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"{""_c109"":"""",""_file_path"":""/Volumes/us_project_catalog/raw/raw_volume/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2026_1.csv""}",2026-09-26T14:37:18.359Z,0,0,0,0,0,0,0,0.0,0.0,1,1,Early,null,null,Unknown
2026,1,01,4,7,2026-01-04,WN,19393,WN,N8948Q,852,12954,1295407,32575,LGB,"Long Beach, CA",CA,06,California,91,14893,1489302,33192,SMF,"Sacramento, CA",CA,06,California,91,1640,1657,17.00,17.00,1.00,1,1600-1659,11.00,1708,1813,3.00,1096.0,1105.6,6.00,6.00,0.00,0,1800-1859,0.00,null,0.00,90.00,79.00,65.00,1.00,387.00,2,null,null,null,null,null,null,null,null,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"{""_c109"":"""",""_file_path"":""/Volumes/us_project_catalog/raw/raw_volume/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2026_1.csv""}",2026-09-26T14:37:18.359Z,0,0,0,0,0,0,0,0.0,0.0,1,1,On Time,null,null,Unknown
2026,1,01,4,7,2026-01-04,WN,19393,WN,N495WN,1253,13204,1320403,31454,MCO,"Orlando, FL",FL,12,Florida,33,10397,1039708,30397,ATL,"Atlanta, GA",GA,13,Georgia,34,1500,1515,15.00,15.00,1.00,1,1500-1559,26.00,1541,1643,5.00,1016.0000000000001,1036.8000000000002,13.00,13.00,0.00,0,1600-1659,0.00,null,0.00,95.00,93.00,62.00,1.00,404.00,2,null,null,null,null,null,null,null,null,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"{""_c109"":"""",""_file_pa

In [0]:
df_bronze_cleaned = try_df

In [0]:
airport_table = "us_project_catalog.raw.Airport_delta_table"
new_airport_df = spark.read.table(airport_table)

broadcast_joined_df = try_df.join(F.broadcast(new_airport_df) , try_df.Origin == new_airport_df.iata, "left")


In [0]:
broadcast_joined_df.display()

year,Quarter,month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,IATA_CODE_Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,OriginAirportID,OriginAirportSeqID,OriginCityMarketID,Origin,OriginCityName,OriginState,OriginStateFips,OriginStateName,OriginWac,DestAirportID,DestAirportSeqID,DestCityMarketID,Dest,DestCityName,DestState,DestStateFips,DestStateName,DestWac,CRSDepTime,DepTime,DepDelay,DepDelayMinutes,DepDel15,DepartureDelayGroups,DepTimeBlk,TaxiOut,WheelsOff,WheelsOn,TaxiIn,CRSArrTime,ArrTime,ArrDelay,ArrDelayMinutes,ArrDel15,ArrivalDelayGroups,ArrTimeBlk,Cancelled,CancellationCode,Diverted,CRSElapsedTime,ActualElapsedTime,AirTime,Flights,Distance,DistanceGroup,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay,FirstDepTime,TotalAddGTime,LongestAddGTime,DivAirportLandings,DivReachedDest,DivActualElapsedTime,DivArrDelay,DivDistance,Div1Airport,Div1AirportID,Div1AirportSeqID,Div1WheelsOn,Div1TotalGTime,Div1LongestGTime,Div1WheelsOff,Div1TailNum,Div2Airport,Div2AirportID,Div2AirportSeqID,Div2WheelsOn,Div2TotalGTime,Div2LongestGTime,Div2WheelsOff,Div2TailNum,Div3Airport,Div3AirportID,Div3AirportSeqID,Div3WheelsOn,Div3TotalGTime,Div3LongestGTime,Div3WheelsOff,Div3TailNum,Div4Airport,Div4AirportID,Div4AirportSeqID,Div4WheelsOn,Div4TotalGTime,Div4LongestGTime,Div4WheelsOff,Div4TailNum,Div5Airport,Div5AirportID,Div5AirportSeqID,Div5WheelsOn,Div5TotalGTime,Div5LongestGTime,Div5WheelsOff,Div5TailNum,_rescued_data,ingested_timestamp,null_flight_date_dq,null_airline_dq,null_origin_dq,null_dest_dq,delay_range_dq,dq_distance_range,cancellation_code_dq,same_origin_dest_dq,Total_flag,dq_is_cleaned,is_Weekend,Delay_cateory,Sum_delay_cause,Data_quality_delay_mismatch,Cancellation_category,altitude,city,country,dst,iata,icao,id,latitude,longitude,name,source,timezone,type,utc_offset,_load_ts
2026,1,01,4,7,2026-01-04,WN,19393,WN,N1803U,3469,12954,1295407,32575,LGB,"Long Beach, CA",CA,06,California,91,11292,1129203,30325,DEN,"Denver, CO",CO,08,Colorado,82,1135,1137,2.00,2.00,0.00,0,1100-1159,17.00,1154,1435,6.00,920.0,905.6,-9.00,0.00,0.00,-1,1400-1459,0.00,null,0.00,135.00,124.00,101.00,1.00,854.00,4,null,null,null,null,null,null,null,null,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"{""_c109"":"""",""_file_path"":""/Volumes/us_project_catalog/raw/raw_volume/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2026_1.csv""}",2026-09-26T14:37:18.359Z,0,0,0,0,0,0,0,0.0,0.0,1,1,Early,null,null,Unknown,60,Long Beach,United States,A,LGB,KLGB,3582,33.81769943,-118.1520004,Long Beach /Daugherty Field/ Airport,OurAirports,America/Los_Angeles,airport,-8,2026-09-10T11:05:41.221Z
2026,1,01,4,7,2026-01-04,WN,19393,WN,N8948Q,852,12954,1295407,32575,LGB,"Long Beach, CA",CA,06,California,91,14893,1489302,33192,SMF,"Sacramento, CA",CA,06,California,91,1640,1657,17.00,17.00,1.00,1,1600-1659,11.00,1708,1813,3.00,1096.0,1105.6,6.00,6.00,0.00,0,1800-1859,0.00,null,0.00,90.00,79.00,65.00,1.00,387.00,2,null,null,null,null,null,null,null,null,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"{""_c109"":"""",""_file_path"":""/Volumes/us_project_catalog/raw/raw_volume/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2026_1.csv""}",2026-09-26T14:37:18.359Z,0,0,0,0,0,0,0,0.0,0.0,1,1,On Time,null,null,Unknown,60,Long Beach,United States,A,LGB,KLGB,3582,33.81769943,-118.1520004,Long Beach /Daugherty Field/ Airport,OurAirports,America/Los_Angeles,airport,-8,2026-09-10T11:05:41.221Z
2026,1,01,4,7,2026-01-04,WN,19393,WN,N495WN,1253,13204,1320403,31454,MCO,"Orlando, FL",FL,12,Florida,33,10397,1039708,30397,ATL,"Atlanta, GA",GA,13,Georgia,34,1500,1515,1

In [0]:
df_with_dest = broadcast_joined_df.join(
    F.broadcast(new_airport_df.select(
        F.col("iata").alias("dest_iata"),
        F.col("latitude").alias("dest_lat"),
        F.col("longitude").alias("dest_lon")
    )),
    F.upper(F.trim(broadcast_joined_df.Dest)) == F.upper(F.trim(F.col("dest_iata"))),
    "left"
)

In [0]:
#Haversine formula

df_with_distance = df_with_dest.withColumn(
    "distance_km",
    2 * 6371 * F.asin(
        F.sqrt(
            F.pow(F.sin(F.radians(F.col("dest_lat").cast("double") - F.col("latitude").cast("double")) / 2), 2) +
            F.cos(F.radians(F.col("latitude").cast("double"))) * F.cos(F.radians(F.col("dest_lat").cast("double"))) *
            F.pow(F.sin(F.radians(F.col("dest_lon").cast("double") - F.col("longitude").cast("double")) / 2), 2)
        )
    )
)



In [0]:
df_bronze_cleaned = df_with_distance

In [0]:
df_bronze_cleaned.display()

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5335769520533506>, line 1
----> 1 df_bronze_cleaned.display()

NameError: name 'df_bronze_cleaned' is not defined

In [0]:


TABLE_NAME = "us_project_catalog.raw.Flight_delta_table"
df_bronze_cleaned.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(TABLE_NAME)   


delta_table = DeltaTable.forName(spark, TABLE_NAME)


spark.read.table(TABLE_NAME).show()

+----+-------+-----+----------+---------+----------+-----------------+------------------------+---------------------------+-----------+-------------------------------+---------------+------------------+------------------+------+-----------------+-----------+---------------+---------------+---------+-------------+----------------+----------------+----+---------------+---------+-------------+-------------+-------+----------+-------+--------+---------------+--------+--------------------+----------+-------+---------+--------+------+------------------+------------------+--------+---------------+--------+------------------+----------+---------+----------------+--------+--------------+-----------------+-------+-------+--------+-------------+------------+------------+--------+-------------+-----------------+------------+-------------+---------------+------------------+--------------+--------------------+-----------+-----------+-----------+-------------+----------------+------------+-----------

In [0]:
spark.read.table("us_project_catalog.raw.Flight_delta_table").show()

+----+-------+-----+----------+---------+----------+-----------------+------------------------+---------------------------+-----------+-------------------------------+---------------+------------------+------------------+------+-----------------+-----------+---------------+---------------+---------+-------------+----------------+----------------+----+---------------+---------+-------------+-------------+-------+----------+-------+--------+---------------+--------+--------------------+----------+-------+---------+--------+------+------------------+------------------+--------+---------------+--------+------------------+----------+---------+----------------+--------+--------------+-----------------+-------+-------+--------+-------------+------------+------------+--------+-------------+-----------------+------------+-------------+---------------+------------------+--------------+--------------------+-----------+-----------+-----------+-------------+----------------+------------+-----------